<a href="https://colab.research.google.com/github/ShahSayem/Alzheimer-Disease-Detection/blob/main/2C-OASIS_Alzheimer_ResNet_18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
ninadaithal_imagesoasis_path = kagglehub.dataset_download('ninadaithal/imagesoasis')

print('Data source import complete.')


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import datasets, models, transforms
from PIL import Image
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score
import time # For timing epochs

# --- Device Configuration ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
base_path = '/kaggle/input/imagesoasis/Data/'

non_demented_paths = []
very_mild_demented_paths = []
mild_demented_paths = []
moderate_demented_paths = []

paths_dict = {
    "Non Demented": non_demented_paths,
    "Very mild Dementia": very_mild_demented_paths,
    "Mild Dementia": mild_demented_paths,
    "Moderate Dementia": moderate_demented_paths
}

for class_name, path_list in paths_dict.items():
    class_dir = os.path.join(base_path, class_name)
    if os.path.isdir(class_dir):
        for filename in os.listdir(class_dir):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                path_list.append(os.path.join(class_dir, filename))
    else:
        print(f"Warning: Directory not found {class_dir}")


print(f"Non Demented: {len(non_demented_paths)}")
print(f"Mild Demented: {len(mild_demented_paths)}")
print(f"Moderate Demented: {len(moderate_demented_paths)}")
print(f"Very Mild Demented: {len(very_mild_demented_paths)}")

Non Demented: 67222
Mild Demented: 5002
Moderate Demented: 488
Very Mild Demented: 13725


In [ ]:
#Create DataFrame and Label Encoding
all_image_paths = []
all_labels_str = []

# Mapping string labels to integer labels
label_map = {
    "Non Demented": 0,
    "Mild Dementia": 1,
    "Moderate Dementia": 2,
    "Very mild Dementia": 3
}

# Create a reverse map for display purposes
idx_to_class = {v: k for k, v in label_map.items()}


for label_str, paths in paths_dict.items():
    if label_str in label_map: # Ensure the class name from directory is in our map
        label_int = label_map[label_str]
        for path in paths:
            all_image_paths.append(path)
            all_labels_str.append(label_str) # Keep string label for df inspection
    else:
        print(f"Warning: Class name '{label_str}' from directory structure not in label_map.")


df = pd.DataFrame({
    'image_path': all_image_paths,
    'label_str': all_labels_str # String label for inspection
})

# Add integer labels based on the map
df['label_int'] = df['label_str'].map(label_map)

# Drop rows where label_int might be NaN if a label_str was not in label_map
df.dropna(subset=['label_int'], inplace=True)
df['label_int'] = df['label_int'].astype(int)


print("\nDataFrame Head:")
print(df.head())
print("\nClass Distribution in DataFrame:")
print(df['label_str'].value_counts())


# --- PyTorch Custom Dataset ---
class AlzheimerDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]['image_path']
        # label is the integer representation
        label = self.dataframe.iloc[idx]['label_int']

        try:
            image = Image.open(img_path).convert('RGB') # Ensure 3 channels for ResNet
        except FileNotFoundError:
            print(f"Error: Image not found at {img_path}")
            # Return a placeholder or handle appropriately
            return torch.randn(3, 224, 224), torch.tensor(-1) # Placeholder
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            return torch.randn(3, 224, 224), torch.tensor(-1) # Placeholder


        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


DataFrame Head:
                                          image_path     label_str  label_int
0  /kaggle/input/imagesoasis/Data/Non Demented/OA...  Non Demented          0
1  /kaggle/input/imagesoasis/Data/Non Demented/OA...  Non Demented          0
2  /kaggle/input/imagesoasis/Data/Non Demented/OA...  Non Demented          0
3  /kaggle/input/imagesoasis/Data/Non Demented/OA...  Non Demented          0
4  /kaggle/input/imagesoasis/Data/Non Demented/OA...  Non Demented          0

Class Distribution in DataFrame:
label_str
Non Demented          67222
Very mild Dementia    13725
Mild Dementia          5002
Moderate Dementia       488
Name: count, dtype: int64


In [ ]:
# --- Image Transformations for ResNet ---
# ResNet models were typically trained on ImageNet with 224x224 images
# and specific normalization.
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(256), # Resize smaller edge to 256
        transforms.CenterCrop(224), # Crop to 224x224
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # Imagenet stats
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# --- Splitting Data ---
if df.empty:
    print("DataFrame is empty. Cannot proceed with splitting and training.")
else:
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_int'])
    # Further split train_df into train and validation for PyTorch loop
    train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df['label_int'])


    print(f"Training samples: {len(train_df)}")
    print(f"Validation samples: {len(val_df)}")
    print(f"Test samples: {len(test_df)}")

    train_dataset = AlzheimerDataset(train_df, transform=data_transforms['train'])
    val_dataset = AlzheimerDataset(val_df, transform=data_transforms['val'])
    test_dataset = AlzheimerDataset(test_df, transform=data_transforms['val']) # Use 'val' transform for test

    BATCH_SIZE = 32
    # NUM_WORKERS can be os.cpu_count() or similar for faster data loading
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=min(4, os.cpu_count() if os.cpu_count() else 1))
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=min(4, os.cpu_count() if os.cpu_count() else 1))
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=min(4, os.cpu_count() if os.cpu_count() else 1))

    dataloaders = {'train': train_loader, 'val': val_loader}
    dataset_sizes = {'train': len(train_dataset), 'val': len(val_dataset)}

Training samples: 55319
Validation samples: 13830
Test samples: 17288


In [ ]:
# ResNet Model Setup
num_classes = len(df['label_int'].unique()) # Should be 4
print(f"Number of classes: {num_classes}")

try:
    model_resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
except TypeError: # Fallback for older torchvision versions
    print("Using older torchvision pretrained=True method for ResNet18.")
    model_resnet = models.resnet18(pretrained=True)


# Freeze all layers except the final one for feature extraction
# for param in model_resnet.parameters():
#     param.requires_grad = False

# Get the number of input features for the classifier
num_ftrs = model_resnet.fc.in_features

# Replace the last fully connected layer
model_resnet.fc = nn.Linear(num_ftrs, num_classes)

model_resnet = model_resnet.to(device)

# Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
# only parameters of final layer are being optimized as
# opposed to before.
# optimizer_ft = optim.Adam(model_resnet.fc.parameters(), lr=0.001) # If only training the classifier
optimizer_ft = optim.Adam(model_resnet.parameters(), lr=0.0001) # Fine-tune all parameters

# Decay LR by a factor of 0.1 every 7 epochs
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=7, gamma=0.1)

Number of classes: 4


In [ ]:
# Training Function
def train_model(model, criterion, optimizer, scheduler, num_epochs=25):
    since = time.time()

    best_model_wts = model.state_dict() # Keep a copy of the best model's weights
    best_acc = 0.0

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data.
            progress_bar = tqdm(dataloaders[phase], desc=f"{phase.capitalize()} Epoch {epoch+1}")
            for inputs, labels in progress_bar:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward
                # Track history if only in train
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                progress_bar.set_postfix(loss=loss.item(), acc=torch.sum(preds == labels.data).item()/inputs.size(0))


            if phase == 'train':
                scheduler.step() # Step the learning rate scheduler

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Store history
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.item()) # .item() to get Python number
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())


            # Deep copy the model if it's the best validation accuracy so far
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = model.state_dict()
                torch.save(model.state_dict(), 'best_resnet_model.pth') # Save best model
                print(f"New best validation accuracy: {best_acc:.4f}, model saved.")


        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')

    # Load best model weights
    model.load_state_dict(best_model_wts)
    return model, history

In [ ]:
# Start Training (if data is loaded)
if not df.empty and num_classes > 0:
    print("\nStarting ResNet model training...")
    # PyTorch loop needs explicit epoch count. It can be adjust later.
    model_resnet_trained, history = train_model(model_resnet, criterion, optimizer_ft, exp_lr_scheduler, num_epochs=15) # e.g., 15 epochs
else:
    print("Skipping training as DataFrame is empty or num_classes is not set.")
    model_resnet_trained = None
    history = None


# Plot Training History
if history:
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.title('Accuracy over Epochs')

    plt.subplot(1, 2, 2)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Loss over Epochs')
    plt.tight_layout()
    plt.show()


Starting ResNet model training...
Epoch 1/15
----------


Train Epoch 1: 100%|██████████| 1729/1729 [03:39<00:00,  7.88it/s, acc=1, loss=0.068]     


train Loss: 0.3280 Acc: 0.8649


Val Epoch 1: 100%|██████████| 433/433 [00:38<00:00, 11.14it/s, acc=1, loss=0.0114]    


val Loss: 0.1741 Acc: 0.9330
New best validation accuracy: 0.9330, model saved.

Epoch 2/15
----------


...

Train Epoch 10: 100%|██████████| 1729/1729 [03:07<00:00,  9.21it/s, acc=1, loss=0.000135]  


train Loss: 0.0021 Acc: 0.9996


Val Epoch 10: 100%|██████████| 433/433 [00:28<00:00, 14.98it/s, acc=1, loss=6.56e-6]   


val Loss: 0.0010 Acc: 0.9998
New best validation accuracy: 0.9998, model saved.

Epoch 11/15
----------


Train Epoch 11:  97%|█████████▋| 1679/1729 [03:02<00:04, 10.03it/s, acc=1, loss=0.000868]  

In [ ]:
# Evaluation on Test Set
def evaluate_model(model, dataloader):
    model.eval() # Set model to evaluation mode
    all_preds = []
    all_labels = []

    with torch.no_grad(): # No need to track gradients during evaluation
        for inputs, labels in tqdm(dataloader, desc="Evaluating Test Set"):
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return np.array(all_labels), np.array(all_preds)

if model_resnet_trained and not df.empty: # Check if model was trained
    print("\nEvaluating on the test set...")
    # Load the best saved model weights for final evaluation
    try:
        model_resnet_trained.load_state_dict(torch.load('best_resnet_model.pth'))
        print("Loaded best model weights for evaluation.")
    except FileNotFoundError:
        print("Warning: 'best_resnet_model.pth' not found. Evaluating with current model weights.")


    y_true_test, y_pred_test = evaluate_model(model_resnet_trained, test_loader)

    test_accuracy = accuracy_score(y_true_test, y_pred_test)
    print(f"\nTest Accuracy: {test_accuracy:.4f}")

    print("\nClassification Report (Test Set):")
    # Use integer labels for report if idx_to_class is available
    target_names_report = [idx_to_class[i] for i in sorted(idx_to_class.keys())]
    print(classification_report(y_true_test, y_pred_test, target_names=target_names_report))

    print("\nConfusion Matrix (Test Set):")
    cm = confusion_matrix(y_true_test, y_pred_test)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names_report)
    disp.plot(cmap=plt.cm.Blues)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("Skipping evaluation as the model was not trained or DataFrame was empty.")